[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/block2.ipynb)

# Block 2: fitting a model, and finding out whether it worked

Work through this before block 2, on your own or in pairs. It takes about
forty minutes. Every answer is written underneath the question, so you cannot
get stuck.

If a cell fails, read the message, then run the cell above it again.

## 1. A real table that ships with scikit-learn

569 rows. Each row is one tissue sample, each column is one measurement, and
the label says which of two kinds it is: malignant, which is a cancer, or
benign. Nothing is downloaded: this table is inside the scikit-learn you
already have.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

raw = load_breast_cancer()
X = pd.DataFrame(raw.data, columns=raw.feature_names)
y = raw.target

print("rows and columns:", X.shape)
print("labels: ", pd.Series(y).value_counts().to_dict())
print("0 means", raw.target_names[0] + ",", "1 means", raw.target_names[1])
X.iloc[:5, :4]

## 2. Hide some of it before you start

Seven rows in ten to learn from, three to check on. `random_state` fixes the
shuffle so everybody gets the same split.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

print("learning from:", X_train.shape[0], "rows")
print("held back:    ", X_test.shape[0], "rows")

## 3. Two models, fitted

A logistic regression, which is the line that answers yes or no, and a
decision tree. Three lines each.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

line = LogisticRegression(max_iter=5000).fit(X_train, y_train)
tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)

for name, m in [("logistic regression", line), ("decision tree", tree)]:
    print("%-20s learned from %.3f   held back %.3f" % (
        name, m.score(X_train, y_train), m.score(X_test, y_test)))

Look at the tree's two numbers. It is perfect on the rows it learned from and
worse on the rows it never saw. That gap is overfitting.

## 4. All four numbers

Accuracy is one number and it hides which mistake was made. The four numbers
say it properly.

In [ ]:
from sklearn.metrics import confusion_matrix

for name, m in [("logistic regression", line), ("decision tree", tree)]:
    grid = confusion_matrix(y_test, m.predict(X_test))
    print(name)
    print("   malignant called malignant:", grid[0, 0],
          "   malignant called benign:", grid[0, 1])
    print("   benign called benign:      ", grid[1, 1],
          "   benign called malignant:", grid[1, 0])

A malignant sample called benign is the expensive mistake: a cancer the model
missed. A benign sample called malignant is a false alarm, which a second test
can clear. Accuracy adds the two together and never showed you which one a
model makes.

In [ ]:
import matplotlib.pyplot as plt

grid = confusion_matrix(y_test, line.predict(X_test))
words = [["malignant called malignant", "malignant called benign:\na missed cancer"],
         ["benign called malignant:\na false alarm", "benign called benign"]]

fig, ax = plt.subplots(figsize=(6, 3.6))
for i in range(2):
    for j in range(2):
        missed = (i, j) == (0, 1)
        ink = "white" if missed else "#1e3a5f"
        ax.add_patch(plt.Rectangle((j, i), 1, 1, edgecolor="white", lw=3,
                                   facecolor="#b3202c" if missed else "#dfe6ee"))
        ax.text(j + 0.5, i + 0.4, grid[i, j], ha="center", va="center",
                fontsize=22, color=ink)
        ax.text(j + 0.5, i + 0.76, words[i][j], ha="center", va="center",
                fontsize=9, color=ink)
ax.set_xlim(0, 2)
ax.set_ylim(2, 0)
ax.set_xticks([0.5, 1.5], ["said malignant", "said benign"])
ax.set_yticks([0.5, 1.5], ["was malignant", "was benign"])
ax.xaxis.tick_top()
ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("logistic regression, on the %d rows held back" % len(y_test),
             pad=24)
plt.tight_layout()
plt.show()

## 5. What if you hid the wrong rows?

One split gives one score, and that score depends on which rows you happened
to hide. Five folds gives five scores.

In [ ]:
from sklearn.model_selection import cross_val_score

folds = {}
for name, m in [("logistic regression", LogisticRegression(max_iter=5000)),
                ("decision tree", DecisionTreeClassifier(random_state=0))]:
    folds[name] = cross_val_score(m, X, y, cv=5)
    print("%-20s %s   average %.3f" % (
        name, " ".join("%.3f" % s for s in folds[name]), folds[name].mean()))

Five scores for each model, one for each choice of which rows to hide. The
tree's average is below the logistic regression's, and so is its best fold.
A single split gives one score from this range, and where in the range it
lands depends on the rows you happened to hide.

## 6. Twenty trees, and a vote

One tree overfits. Twenty trees, each grown on a different part of the data,
voting on the answer, is a random forest.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=20, random_state=0)
folds["random forest"] = cross_val_score(forest, X, y, cv=5)

print("%-20s %s   average %.3f" % ("random forest", " ".join(
    "%.3f" % s for s in folds["random forest"]), folds["random forest"].mean()))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.6))
for row, (name, scores) in enumerate(folds.items()):
    ax.scatter(scores, [row] * len(scores), color="#1e3a5f", s=36, zorder=3)
    ax.plot([scores.mean()] * 2, [row - 0.3, row + 0.3], color="#b45309", lw=3)
ax.set_yticks(range(len(folds)), list(folds))
ax.invert_yaxis()
ax.set_xlabel("share of the hidden rows predicted right")
ax.set_title("one dot per fold, and the average as an orange bar")
ax.grid(axis="x", color="#dfe6ee")
plt.tight_layout()
plt.show()

The forest scores higher than the single tree on each of the five folds, and
its average is the highest of the three. The trees in a forest make different
mistakes, and the vote outnumbers many of them.

## 7. Your turn

Change `n_estimators=20` to `n_estimators=200` and run the cell again. Then
change `test_size=0.3` to `test_size=0.5` in section 2 and run everything
below it.

What to expect. Ten times as many trees moves the forest's average by less
than a point: past a few dozen trees, more trees change little. Holding back
half the rows moves the two held-back scores by under two points, and the
tree's goes up. The five-fold scores and their chart do not move at all,
because cross-validation scores the whole table whichever split you chose.

Bring one number you did not expect to block 2.